#### <span style="color:cyan">**Imports**</span>

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd

from util.data_preprocessing import *
from util.pca_plotting import *
from util.classes.PCADataset import *
from util.classes.PcaMLP import *
from util.model_eval import *
from util.model_optimization import *

from preprocessing.pipelines import build_pca_pipeline, build_peak_pipeline
import json
import os
import math
import numpy as np
import torch

#### <span style="color:cyan">**Data Preprocessing**</span>

In [ ]:
# Config
pd.set_option('display.max_columns', 10)
DATA_PATH = '../data/batch2'
DOMAIN = (10.25, 11.0)
TEST_IDX = np.arange(0, 1024, 1)
INTERP = 4
WINDOW = 20
THRESHOLD = 0.2
K = 10
K_SWEEP = [2, 5, 10, 20, 40, 80]
N_PC_SHOW = 6

# Import Data
geom_labels, geom_table, refl_data, abs_data, refl_bg, abs_bg, wl = import_data(DATA_PATH)

# Augment geometry table from 4D -> 8D
geom_table_8d = derive_geometry_features(geom_table)
geom_labels_8d = geom_table_8d.columns.to_numpy()

# Print Data and geometry Table for inspection
# print(f'\ngeometry labels (8D):\n{geom_labels_8d}\n')
# print(f'geometry_table (8D):\n{geom_table_8d.head()}\n')
# print(f'absorption data:\n{abs_data.head()}\n')
# print(f'absorption background:\n{abs_bg.head()}\n')

# Build pipelines
abs_pipe_peak = build_peak_pipeline(DOMAIN, INTERP, WINDOW, THRESHOLD, TEST_IDX, background=abs_bg)
abs_pipe_pca  = build_pca_pipeline(K, DOMAIN, INTERP, background=abs_bg)

# Fit pipelines and extract relevant data
abs_ft_pca  = abs_pipe_pca.fit_transform(abs_data) # PCA coefficients
abs_ft_peak = abs_pipe_peak.fit_transform(abs_data) # Peak Data (lambda, gamma, A)
abs_pca  = abs_pipe_pca.named_steps['pca'].artifacts_ # PCA artifacts
abs_data  = abs_pipe_pca[:-1].fit_transform(abs_data) # Spectrum just before PCA
wl = abs_data.columns.to_numpy(dtype=float) # wavelength axis

# Print Feature tables for inspection
print(f'PCA Feature Table (abs):\n{abs_ft_pca.head()}\n')
print(f'Peak Feature Table (abs):\n{abs_ft_peak.head()}\n')


# 1) Scree / cumulative explained variance
plot_scree(abs_pca, 'Absorption', k_max=20)

# 2) Shapes of the first N principal components
plot_pc_shapes(abs_pca, 'Absorption', N_PC_SHOW)

# 3) Reconstruction fidelity vs K on a single example spectrum
demo_idx = int(TEST_IDX[len(TEST_IDX) // 2])
plot_k_sweep_reconstruction(abs_data, abs_pca, demo_idx, K_SWEEP, 'Absorption')

# 4) Spectral MSE and peak-location MAE vs K (dataset-wide diagnostic)
plot_k_sweep_error(abs_data, abs_pca, K_SWEEP, 'Absorption')

# 5) Latent space scatter: PC0 vs PC1 colored by each geometry parameter (all 8).
plot_latent_colored_by_geom(abs_ft_pca, geom_table_8d, geom_labels_8d, 'Absorption')

# 6) Reconstruction grid: data (target) vs K-component PCA reconstruction
#    for each sample in TEST_IDX. Direct visual check of PCA fidelity.
plot_reconstruction_grid(abs_data, abs_pca, TEST_IDX, 'Absorption')


# Data dictionaries for later
pca_ft = {
    "wl": wl,
    "absorption": abs_data,
    "absorption_pca_features": abs_ft_pca,
    "absorption_peak_features": abs_ft_peak,
    "absorption_pca": abs_pca,
    "geometry_labels": geom_labels_8d,
    "geometry_table": geom_table_8d,
    "K": K,
}

data_config = {
    "data_path": DATA_PATH,
    "domain": DOMAIN,
    "interpolation_density": INTERP,
    "fit_window": WINDOW,
    "fit_threshold": THRESHOLD,
    "K": K,
    "geometry_dim": 8,
    "derived_features": ["width_sum", "width_diff", "aspect_ratio", "fill_factor"],
}

KeyboardInterrupt: 

: 